# 01 · Loading and cleaning ZTF transient data with pandas

**Series:** Exploring the ZTF Bright Transient Survey to find Type Ia supernovae
1. **Loading & cleaning with pandas** ← you are here
2. Visualising with matplotlib & seaborn
3. Finding patterns & a first Ia classifier

---

### The data

The **Zwicky Transient Facility (ZTF)** is a wide-field camera on the 48-inch telescope at Palomar Observatory. It scans the whole northern sky every couple of nights and flags anything that changed brightness: supernovae, variable stars, flaring black holes and more.

The **Bright Transient Survey (BTS)** is the ZTF programme that tries to take a spectrum of *every* transient that gets brighter than magnitude 18.5. A spectrum is what tells astronomers what an object really is ("SN Ia", "SN II", "CV", ...). Because the survey is close to complete, it's the best labelled dataset available for training a classifier.

`bts.csv` is an export from the BTS Sample Explorer: **one row per transient**, with summary numbers describing its light curve (how it brightened and faded) and, if it has one, its spectroscopic classification.

### What this notebook covers
| pandas skill | Where it is used |
|---|---|
| `read_csv`, `head`, `info`, `describe` | First look at the data |
| `na_values`, `isna` | Missing values written as `-` |
| `loc`, `iloc`, boolean masks, `query` | Selecting rows and columns |
| `.str` accessor, `to_numeric` | Values like `>5.6` (lower limits) |
| `to_datetime`, `.dt` accessor, `to_timedelta` | Dates in mixed formats, Julian dates |
| `duplicated`, `drop_duplicates` | The same object listed twice |
| `map`, `groupby`, `agg`, `crosstab` | Grouping 39 labels into useful classes |
| `to_parquet` | Saving the clean table for the next notebooks |

We'll also use **astropy**, the standard Python astronomy library, to handle sky coordinates.

## 0 · Setup

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)   # show every column instead of "..."
pd.set_option("display.width", 200)
print("pandas", pd.__version__)

pandas 3.0.6


## 1 · First look

`pd.read_csv` reads a CSV file into a **DataFrame**, which is a table with labelled columns. Each column is a **Series**: a one-dimensional array with a single data type (`dtype`).

In [2]:
raw = pd.read_csv("bts.csv")
raw.shape          # (rows, columns)

(11217, 19)

In [5]:
raw.head()        # first 5 rows; raw.tail() shows the last 5, raw.sample(5) 5 random ones

,ZTFID,IAUID,discID,discdate,RA,Dec,peakt,peakfilt,peakmag,peakabs,duration,rise,fade,type,redshift,hostabs,hostcol,b,A_V
0,ZTF17aaaeqms,AT2023jkf,NaN,2023-05-25T02:51:57.285431,01:19:13.43,+31:46:26.1,2102.97,r,17.5839,-,18.001,12.004,5.997,-,-,-,-,-30.726791,0.205
1,ZTF17aaahrni,AT2016dxt,Gaia16als,2016-04-20 06:27:21.000,05:21:41.58,-04:11:10.4,1155.90,g,17.1049,-,7.667,0.86,6.807,CV,-,-,-,-21.841960,0.827
2,ZTF17aaajowi,-,NaN,2021-05-28,11:47:44.35,+14:34:10.4,1362.73,r,18.9963,-,>5.611,2.651,>2.96,-,-,-,-,70.564727,0.127
3,ZTF17aaaukqn,AT2018fsl,NaN,2018-09-03,01:48:53.10,+54:21:44.5,364.91,g,17.1019,-,8.202,1.215,6.987,CV,-,-,-,-7.571489,0.484
4,ZTF17aaawgbm,AT2019pla,ZTF17aaawgbm,2019-09-04 06:17:16.000,19:52:32.99,+44:41:20.0,2537.80,r,17.0274,-,>7.353,>0,>7.353,CV,-,-,-,8.855999,0.949


In [6]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 11217 entries, 0 to 11216
Data columns (total 19 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ZTFID     11217 non-null  str    
 1   IAUID     11217 non-null  str    
 2   discID    8325 non-null   str    
 3   discdate  11217 non-null  str    
 4   RA        11217 non-null  str    
 5   Dec       11217 non-null  str    
 6   peakt     11217 non-null  float64
 7   peakfilt  11217 non-null  str    
 8   peakmag   11217 non-null  float64
 9   peakabs   11217 non-null  str    
 10  duration  11217 non-null  str    
 11  rise      11217 non-null  str    
 12  fade      11217 non-null  str    
 13  type      11217 non-null  str    
 14  redshift  11217 non-null  str    
 15  hostabs   11217 non-null  str    
 16  hostcol   11217 non-null  str    
 17  b         11217 non-null  float64
 18  A_V       11217 non-null  float64
dtypes: float64(4), str(15)
memory usage: 2.8 MB


**Warning:** columns that should be numbers, like `redshift`, `rise` and `fade`, came in as `str` (text). pandas only treats a column as numeric if *every* value in it parses as a number. Two things get in the way here:

* missing values are written as a dash `-` rather than left empty;
* some times are written as **lower limits**, like `>5.611`.

Before fixing anything, here's what each column means.

## 2 · Column dictionary

| Column | Meaning | Notes |
|---|---|---|
| `ZTFID` | ZTF's internal name for the object | unique per row |
| `IAUID` | Official IAU name: `SN2023abc` if classified as a supernova, otherwise `AT2023abc` ("astronomical transient") | |
| `discID` | Name used by whoever discovered it first, if that was not ZTF (e.g. Gaia, ATLAS) | empty = ZTF found it itself |
| `discdate` | Discovery date | three different formats |
| `RA`, `Dec` | Sky position (right ascension, declination) | sexagesimal text, `hh:mm:ss`, `±dd:mm:ss` |
| `peakt` | Time of peak brightness, in days since Julian Date 2,458,000 | |
| `peakfilt` | Filter in which the peak was measured: `g` (green) or `r` (red) | |
| `peakmag` | Apparent magnitude at peak. **Smaller is brighter.** | log scale: 5 mag = 100× in flux |
| `peakabs` | Absolute magnitude at peak (true luminosity), computed from `peakmag` and `redshift` | needs a redshift |
| `rise` | Days to rise from half of peak flux to peak | can be a lower limit `>` |
| `fade` | Days to fade from peak back to half of peak flux | can be a lower limit `>` |
| `duration` | `rise + fade`: time spent above half of peak flux | can be a lower limit `>` |
| `type` | Spectroscopic classification. **This is the label we want to predict.** | `-` = not classified |
| `redshift` | Redshift *z*, a proxy for distance | usually measured from the classification spectrum |
| `hostabs`, `hostcol` | Host galaxy absolute magnitude and colour | |
| `b` | Galactic latitude in degrees (0 = the plane of the Milky Way) | |
| `A_V` | Dust extinction in our own Galaxy along this line of sight, in magnitudes | |

## 3 · Missing values

How many dashes does each column contain?

In [7]:
(raw == "-").sum()

ZTFID          0
IAUID        107
discID         0
discdate       0
RA             0
Dec            0
peakt          0
peakfilt       0
peakmag        0
peakabs     4060
duration       0
rise           0
fade           0
type        3374
redshift    4059
hostabs     4362
hostcol     4362
b              0
A_V            0
dtype: int64

`raw == "-"` gives a DataFrame of `True`/`False`, and `.sum()` counts the `True` values in each column (`True` counts as 1).

The fix is to tell `read_csv` which strings mean "missing", using `na_values`. pandas will then store them as `NaN` ("not a number"), its marker for missing data.

In [8]:
df = pd.read_csv("bts.csv", na_values="-")
df.dtypes

ZTFID           str
IAUID           str
discID          str
discdate        str
RA              str
Dec             str
peakt       float64
peakfilt        str
peakmag     float64
peakabs     float64
duration        str
rise            str
fade            str
type            str
redshift    float64
hostabs     float64
hostcol     float64
b           float64
A_V         float64
dtype: object

`peakabs`, `redshift`, `hostabs` and `hostcol` are now `float64`. `rise`, `fade` and `duration` are still text because of the `>` signs, which section 5 deals with.

What *fraction* of each column is missing?

In [9]:
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0].round(3)

hostabs     0.389
hostcol     0.389
peakabs     0.362
redshift    0.362
type        0.301
discID      0.258
IAUID       0.010
dtype: float64

Missing values can carry information of their own:

* **`type` is missing for ~30% of objects.** These transients were never classified. They can't be used for training, but they are exactly the objects a classifier would be used on later.
* `redshift` and `peakabs` are missing for ~36%, slightly *more* than `type`. Hold on to that; it turns out to matter a lot in notebook 3.
* A missing `discID` just means ZTF was the discoverer.

Checking whether redshift is ever known when type is missing takes one line, using a **cross-tabulation**, which counts each combination:

In [10]:
pd.crosstab(df["type"].isna(), df["redshift"].isna(),
            rownames=["type missing?"], colnames=["redshift missing?"])

redshift missing?,False,True
type missing?,,
False,7158,685
True,0,3374


**Redshift is never known without a classification.** (The top-right cell is objects that are classified but have no redshift; the bottom-left is zero.) Redshift is measured from the *same spectrum* that classifies the object. Keep that in mind.

## 4 · Selecting data

There are several ways to pick out parts of a DataFrame:

| Syntax | Selects |
|---|---|
| `df["col"]` | one column, as a Series |
| `df[["a", "b"]]` | several columns, as a DataFrame |
| `df.loc[rows, cols]` | by **label** (index values and column names) |
| `df.iloc[rows, cols]` | by **position** (0, 1, 2, ...) |
| `df[mask]` | rows where a boolean Series `mask` is `True` |
| `df.query("...")` | rows matching a condition written as a string |

In [9]:
df.iloc[:3, :6]    # first 3 rows, first 6 columns

,ZTFID,IAUID,discID,discdate,RA,Dec
0,ZTF17aaaeqms,AT2023jkf,NaN,2023-05-25T02:51:57.285431,01:19:13.43,+31:46:26.1
1,ZTF17aaahrni,AT2016dxt,Gaia16als,2016-04-20 06:27:21.000,05:21:41.58,-04:11:10.4
2,ZTF17aaajowi,NaN,NaN,2021-05-28,11:47:44.35,+14:34:10.4


In [10]:
df.loc[df["peakmag"] < 14, ["ZTFID", "IAUID", "type", "peakmag", "redshift"]]

,ZTFID,IAUID,type,peakmag,redshift
18,ZTF18aaaokus,AT2018hvb,AGN,13.8042,NaN
612,ZTF18acbvgqw,SN2018htt,SN Ia,13.9113,0.00867
616,ZTF18acbwaxk,SN2018hna,SN II,13.9583,0.00240
944,ZTF19aacgslb,SN2019np,SN Ia,13.4112,0.00452
1543,ZTF19aaxugwk,AT2019hau,CV?,13.7036,NaN
2722,ZTF20aaelulu,SN2020oi,SN Ic,13.8596,0.00520
3069,ZTF20aavpnlv,SN2020hvf,SN Ia,12.5716,0.00583
3876,ZTF20achlced,SN2020uxz,SN Ia,13.5029,0.00824
4198,ZTF20actzmzp,AT2020abfh,CV,12.3460,NaN
4708,ZTF21aanehlz,AT2021dxi,CV,13.5836,NaN


The brightest objects in the sample fall into two groups: **very nearby supernovae** (redshift below 0.01, so within roughly 150 million light years) and **cataclysmic variables (CVs)**, which are in our own Galaxy and so have no redshift.

Masks can be combined with `&` (and), `|` (or) and `~` (not). **Each condition needs its own parentheses.**

In [11]:
bright_ia = df[(df["type"] == "SN Ia") & (df["peakmag"] < 16)]
len(bright_ia)

93

In [12]:
# The same selection written with query(), which is often easier to read:
df.query("type == 'SN Ia' and peakmag < 16")[["ZTFID", "IAUID", "peakmag", "redshift"]].head()

,ZTFID,IAUID,peakmag,redshift
70,ZTF18aahvndq,SN2020jhf,15.3851,0.01537
107,ZTF18aamzgzi,SN2018mef,15.2602,0.01790
130,ZTF18aaqkcso,SN2019fck,14.4015,0.02000
163,ZTF18aasdted,SN2018big,15.7176,0.01815
235,ZTF18aaxhcam,SN2024fjp,15.5155,0.03143


## 5 · Lower limits: the `>` problem

In [14]:
df[["ZTFID", "rise", "fade", "duration"]].head(8)

,ZTFID,rise,fade,duration
0,ZTF17aaaeqms,12.004,5.997,18.001
1,ZTF17aaahrni,0.86,6.807,7.667
2,ZTF17aaajowi,2.651,>2.96,>5.611
3,ZTF17aaaukqn,1.215,6.987,8.202
4,ZTF17aaawgbm,>0,>7.353,>7.353
5,ZTF17aaazdba,24.318,42.102,66.42
6,ZTF17aabbwwl,7.295,16.125,23.42
7,ZTF17aabtvsy,>6.01,28.219,>34.229


A value like `rise = >5.611` means *"it took **at least** 5.6 days to rise"*. The object was already brightening when ZTF first saw it, usually because it went off while its patch of sky was behind the Sun or the weather was bad. Statisticians call data like this **censored**: we know a bound, not the exact value.

Throwing these rows away would remove a whole group of objects and bias the sample. Instead, each of these columns becomes two:

* the number itself (`5.611`), and
* a `True`/`False` flag saying whether it is only a limit.

A model can use the flag as a feature in its own right.

In [15]:
for col in ["rise", "fade", "duration"]:
    text = df[col]
    df[f"{col}_is_limit"] = text.str.startswith(">")      # .str gives string methods for a whole column
    df[col] = pd.to_numeric(text.str.lstrip(">"))         # remove the ">" and convert to numbers

df[["rise", "rise_is_limit", "fade", "fade_is_limit", "duration", "duration_is_limit"]].head(8)

,rise,rise_is_limit,fade,fade_is_limit,duration,duration_is_limit
0,12.004,False,5.997,False,18.001,False
1,0.860,False,6.807,False,7.667,False
2,2.651,False,2.960,True,5.611,True
3,1.215,False,6.987,False,8.202,False
4,0.000,True,7.353,True,7.353,True
5,24.318,False,42.102,False,66.420,False
6,7.295,False,16.125,False,23.420,False
7,6.010,True,28.219,False,34.229,True


In [16]:
df[["rise_is_limit", "fade_is_limit", "duration_is_limit"]].mean().round(3)   # mean of booleans = fraction True

rise_is_limit        0.262
fade_is_limit        0.184
duration_is_limit    0.348
dtype: float64

About a quarter of rise times and a fifth of fade times are only limits. That's too many to ignore.

In [17]:
df[["peakmag", "rise", "fade", "duration", "redshift", "peakabs", "A_V"]].describe().round(2)

,peakmag,rise,fade,duration,redshift,peakabs,A_V
count,11217.00,11217.00,11217.00,11217.00,7158.00,7157.00,11217.00
mean,18.20,15.74,19.83,35.56,0.06,-18.81,0.19
std,0.78,69.92,56.47,92.76,0.03,1.21,0.18
min,12.35,0.00,0.00,0.00,0.00,-23.59,0.01
25%,17.90,6.94,10.68,19.95,0.04,-19.41,0.07
50%,18.44,9.33,14.26,24.34,0.06,-19.11,0.12
75%,18.76,12.16,18.97,29.48,0.08,-18.49,0.23
max,19.00,2600.41,2413.20,2606.41,0.52,-5.51,1.00


`describe()` gives a quick summary of every numeric column. Two things stand out:

* `peakmag` **never goes above 19.0**. That's the brightness cut used to select objects for this catalogue. Almost everything in it is a *selection effect* of some kind, which notebook 2 shows in plots.
* `rise` has a minimum of 0. Those are objects caught right at peak, nearly all of them flagged as limits.

## 6 · Dates and times

In [18]:
df["discdate"].sample(6, random_state=3)

535         2018-12-29 16:19:37.000
380                      2018-07-19
1945        2019-08-21 11:35:31.000
1992        2019-09-01 14:24:00.000
5623        2021-08-28 05:16:27.998
10968    2026-01-14T15:19:17.710881
Name: discdate, dtype: str

The dates come in three formats: date only, date + time, and ISO format with a `T`. `pd.to_datetime(..., format="mixed")` works out the format for each value separately.

In [19]:
df["discdate"] = pd.to_datetime(df["discdate"], format="mixed")
df["discdate"].dtype

dtype('<M8[us]')

A datetime column has a `.dt` accessor, the date equivalent of `.str`:

In [20]:
df["discdate"].dt.year.value_counts().sort_index()

discdate
2016      20
2017      20
2018     557
2019    1753
2020    1765
2021    1664
2022    1285
2023    1147
2024    1367
2025    1306
2026     333
Name: count, dtype: int64

ZTF started survey operations in March 2018, and 2026 is only partly covered by this export. The handful of 2016–2017 dates belong to objects that someone else discovered first and that ZTF later saw flare again.

### Is every year classified equally?

Now that dates are real dates, we can group by year and check what fraction of objects has a `type`:

In [21]:
df.groupby(df["discdate"].dt.year)["type"].apply(lambda t: t.notna().mean()).round(2)

discdate
2016    0.95
2017    0.95
2018    0.86
2019    0.75
2020    0.73
2021    0.73
2022    0.72
2023    0.67
2024    0.70
2025    0.64
2026    0.01
Name: type, dtype: float64

Coverage is 64–86% for every year **except 2026, where it's almost zero.** Has nothing from 2026 been classified? The IAU names suggest otherwise. On the Transient Name Server an object is renamed from `AT...` to `SN...` *only after* a spectrum confirms it is a supernova:

In [22]:
iau_prefix = df["IAUID"].str[:2]
pd.crosstab(df["discdate"].dt.year, iau_prefix).loc[2018:]

IAUID,AT,SN,TD
discdate,,,
2018,223,333,0
2019,672,1070,0
2020,595,1150,0
2021,497,1122,0
2022,389,870,0
2023,385,759,0
2024,472,895,0
2025,472,826,7
2026,109,224,0


Two thirds of 2026 objects already carry `SN` names, so they **have** been classified. This export simply doesn't include their types yet; it was taken before the BTS table caught up. Two lessons:

1. **Data snapshots lag reality.** In 2026, "missing type" means "not in this file yet" rather than "unclassified".
2. **The IAU name leaks the label.** An `SN` prefix only appears *after* classification, so `IAUID` must **never** be used as a model feature. It would be cheating.

**Peak time.** `peakt` counts days since Julian Date (JD) 2,458,000. Astronomers use JD as a continuous day count, and Rather than look up which calendar date JD 2,458,000 is (and risk getting it wrong), we can ask astropy:

In [24]:
from astropy.time import Time
Time(2458000, format="jd").iso

'2017-09-03 12:00:00.000'

In [25]:
JD_2458000 = pd.Timestamp(Time(2458000, format="jd").datetime)   # 2017-09-03 12:00 UTC
df["peakdate"] = JD_2458000 + pd.to_timedelta(df["peakt"], unit="D")

# Subtracting two datetime columns gives a timedelta column
df["disc_to_peak_days"] = (df["peakdate"] - df["discdate"]).dt.total_seconds() / 86400
df["disc_to_peak_days"].describe().round(1)

count    11217.0
mean        59.4
std        287.2
min      -2227.0
25%          6.8
50%         11.0
75%         15.0
max       3504.4
Name: disc_to_peak_days, dtype: float64

The median is about **11 days**: transients are typically discovered on the rise and peak a week or two later. That's a useful check that the date conversion is right.

The extreme values (thousands of days in either direction) are worth a look. **Always look at your outliers:**

In [26]:
cols = ["ZTFID", "IAUID", "discID", "discdate", "peakdate", "type", "disc_to_peak_days"]
pd.concat([df.nsmallest(4, "disc_to_peak_days")[cols],
           df.nlargest(4, "disc_to_peak_days")[cols]])

,ZTFID,IAUID,discID,discdate,peakdate,type,disc_to_peak_days
3126,ZTF20aaxqzkm,AT2020afjr,NaN,2026-06-05 09:03:55.113592,2020-04-30 08:09:36.000000000,NaN,-2227.037721
1579,ZTF19aaziqbb,AT2019hco,NaN,2023-04-08 22:54:09.744360,2019-06-13 09:07:12.000000000,CV,-1395.574279
416,ZTF18ablookq,AT2018maf,NaN,2022-02-13 13:53:17.916094,2018-08-04 07:26:24.000000000,CV,-1289.268680
5685,ZTF21abzithf,SN2024svu,ZTF21abzithf,2024-08-15 10:07:15.744000,2021-09-20 05:45:36.000000000,SN Ia,-1060.181710
558,ZTF18abxmbfq,AT2016dup,Gaia16ahi,2016-03-03 20:34:04.000000,2025-10-07 05:31:12.000000000,CV,3504.373009
2771,ZTF20aafxzej,AT2017cf,iPTF17cf,2017-01-07 05:55:40.000000,2025-12-02 07:55:11.999999991,AGN,3251.083009
2506,ZTF19acszwgx,AT2016hnn,ATLAS16dos,2016-10-27 04:42:14.000000,2025-09-17 03:21:35.999999991,CV,3246.944005
274,ZTF18aazvajy,AT2016cnq,PS16coe,2016-05-23 13:27:39.000000,2024-11-06 04:47:59.999999982,CV,3088.639132


These are mostly repeating sources: CVs, AGN and other objects that flare again and again. The discovery date refers to one outburst and the peak to another. The odd supernova in the list is probably a bookkeeping mix-up, where the ZTF ID was created for an earlier flare at the same position. It's a reminder that "one row = one explosion" is only reliably true for supernovae.

## 7 · Sky coordinates with astropy

`RA` and `Dec` are text like `05:21:41.58` / `-04:11:10.4`. Plotting or calculating with them needs decimal degrees. astropy's `SkyCoord` parses these strings and converts between coordinate systems.

In [27]:
from astropy.coordinates import SkyCoord
import astropy.units as u

coords = SkyCoord(df["RA"], df["Dec"], unit=(u.hourangle, u.deg))
df["ra_deg"] = coords.ra.deg
df["dec_deg"] = coords.dec.deg
df[["RA", "ra_deg", "Dec", "dec_deg"]].head(3)

,RA,ra_deg,Dec,dec_deg
0,01:19:13.43,19.805958,+31:46:26.1,31.773917
1,05:21:41.58,80.423250,-04:11:10.4,-4.186222
2,11:47:44.35,176.934792,+14:34:10.4,14.569556


### Checking a column instead of trusting it

The file already has `b`, the galactic latitude. Since we have coordinates, we can **recompute** it and compare:

In [28]:
df["b_check"] = coords.galactic.b.deg
diff = (df["b"] - df["b_check"]).abs()
print(f"Rows where b disagrees by more than 0.01 deg: {(diff > 0.01).sum()}")

Rows where b disagrees by more than 0.01 deg: 33


In [29]:
df.loc[diff > 0.01, ["ZTFID", "IAUID", "RA", "Dec", "b", "b_check", "A_V", "discdate"]].head(10)

,ZTFID,IAUID,RA,Dec,b,b_check,A_V,discdate
690,ZTF18acjpret,SN2022prr,19:01:41.89,+40:45:03.7,54.535723,15.514393,0.072,2022-07-27 07:12:00.000000
909,ZTF19aaagfme,SN2022okv,23:19:48.11,+07:06:02.2,54.535723,-49.212458,0.072,2022-07-06 13:06:04.896000
6388,ZTF22aaofpts,SN2022mto,22:59:40.81,-09:33:28.7,33.482896,-58.326426,0.205,2022-06-13 10:52:24.997000
6431,ZTF22aapbmfp,AT2022aexp,17:24:05.05,-11:35:32.8,54.535723,13.455180,0.072,2022-07-02 15:45:48.192546
6437,ZTF22aapkbkl,SN2022odp,23:45:33.32,+29:38:25.9,54.535723,-31.082878,0.072,2022-07-03 10:50:51.936000
6439,ZTF22aapmmtu,SN2022ojj,18:29:31.71,+35:05:24.3,54.535723,19.368343,0.072,2022-07-06 08:23:00.384000
6440,ZTF22aapqaqe,SN2022npv,18:53:28.60,+68:28:31.4,54.535723,24.915348,0.072,2022-06-26 10:43:35.996000
6443,ZTF22aaqhirm,SN2022ohi,17:33:45.11,+17:36:24.9,54.535723,24.834565,0.072,2022-06-29 06:36:02.998000
6444,ZTF22aaqiizy,AT2022oep,23:07:20.50,+78:38:41.6,54.535723,16.833750,0.072,2022-07-05 00:00:00.000000
6445,ZTF22aaqlfwe,SN2022oco,21:27:38.35,+00:34:13.8,54.535723,-33.788361,0.072,2022-06-28 10:02:42.003000


In [30]:
print("Size of the disagreement in bad rows (deg):", diff[diff > 0.01].min().round(1), "to", diff[diff > 0.01].max().round(1))
print("Largest disagreement in all other rows (deg):", diff[diff <= 0.01].max().round(4))
df.loc[diff > 0.01].groupby(["b", "A_V"]).size().sort_values(ascending=False)

Size of the disagreement in bad rows (deg): 14.6 to 130.1
Largest disagreement in all other rows (deg): 0.0005


b           A_V  
 54.535723  0.072    23
 52.522920  0.037     2
-37.594027  0.122     1
-49.709965  0.318     1
-37.329998  0.220     1
-37.253759  0.092     1
 38.626809  0.095     1
 33.482896  0.205     1
 40.769949  0.029     1
 82.030487  0.047     1
dtype: int64

**We found a data error.** The 33 bad rows are off by 15° to 130°, while every other row agrees to within 0.0005°, so these are clearly wrong rather than rounding differences.

* **23 objects** at completely different positions all have *the same* `b` (54.54°) and *the same* `A_V` (0.072). All of them were discovered between June and July 2022, which suggests a glitch in whatever pipeline filled these columns at the time.
* The other **10** have values that, as far as we can tell, were copied from *other* objects in the table. Some of these `b` values appear twice in the raw file.

Each value on its own looks perfectly reasonable. The problem only shows up when you cross-check.

The fix:
* `b`: replace with the recomputed value, which is correct for every row;
* `A_V`: recomputing it needs a Galactic dust map, so set the 33 bad values to missing rather than keep wrong numbers.

In [31]:
bad_position_data = diff > 0.01
df["b"] = df["b_check"]
df.loc[bad_position_data, "A_V"] = np.nan
df = df.drop(columns="b_check")
print("A_V now missing for", df["A_V"].isna().sum(), "rows")

A_V now missing for 33 rows


## 8 · Duplicates

`ZTFID` should be unique, and it is. But can the same physical object appear under two different ZTF IDs?

In [32]:
print("ZTFID unique?", df["ZTFID"].is_unique)

named = df["IAUID"].notna()
dupes = df[named & df["IAUID"].duplicated(keep=False)].sort_values("IAUID")
print("Rows sharing an IAU name:", len(dupes))
dupes[["ZTFID", "IAUID", "RA", "Dec", "type", "peakmag", "discdate"]].head(10)

ZTFID unique? True
Rows sharing an IAU name: 26


,ZTFID,IAUID,RA,Dec,type,peakmag,discdate
601,ZTF18acaisao,AT2019lla,23:54:39.16,+28:22:50.1,CV,15.9397,2019-07-14 10:49:26.000000
892,ZTF18adhzhfw,AT2019lla,23:54:39.16,+28:22:50.1,CV,15.9397,2020-11-07 05:01:36.707579
2300,ZTF19acgcrhb,AT2019tee,03:57:28.00,+35:37:16.1,CV?,16.7681,2019-10-19 08:41:16.000000
2632,ZTF19adbkvnr,AT2019tee,03:57:28.00,+35:37:16.1,CV,16.4380,2019-10-19 08:41:16.000000
4144,ZTF20acrwxgr,AT2020aahd,01:38:00.87,+05:57:14.4,NaN,18.6998,2020-11-25 00:00:00.000000
4156,ZTF20acshkpw,AT2020aahd,01:38:00.87,+05:57:14.3,NaN,18.9060,2020-11-26 00:00:00.000000
4271,ZTF20acvglwq,AT2020abss,22:22:42.69,-03:03:45.3,NaN,17.6106,2020-12-11 00:00:00.000000
9202,ZTF24abikdmj,AT2020abss,22:22:42.70,-03:03:44.9,NaN,17.6106,2024-09-24 06:37:41.021368
4612,ZTF21aajglah,AT2021dra,12:06:47.16,+69:07:49.2,NaN,18.8040,2021-02-25 18:48:05.262805
4712,ZTF21aanncjl,AT2021dra,12:06:47.16,+69:07:49.2,NaN,18.8040,2021-02-27 00:00:00.000000


Yes: 13 objects appear twice, at the same position and with the same IAU name. ZTF sometimes gives an object a second ID, for example when it is detected again years later.

If we kept both rows, the same object would count twice, and could even end up in both the training and the test set, which would make the model look better than it is. We'll keep **one row per IAU name**, preferring a row that has a classification:

In [33]:
# Sort so that, within each IAU name, rows with a type come first (NaN goes last),
# then keep the first row of each name. Rows without an IAU name are all kept.
df = df.sort_values(["IAUID", "type"], na_position="last")
is_repeat = df["IAUID"].notna() & df["IAUID"].duplicated(keep="first")
df = df[~is_repeat].sort_index()
print("Rows after de-duplication:", len(df))

Rows after de-duplication: 11204


## 9 · From 39 labels to useful classes

In [34]:
df["type"].value_counts(dropna=False)

type
SN Ia              5062
NaN                3370
SN II               901
CV                  389
SN IIn              178
AGN                 171
SN Ic               153
SN Ia-91T           151
SN Ib               114
SN IIb              108
CV?                  91
SN Ic-BL             70
SLSN-I               65
SN IIP               62
SN Ia-91bg           40
SN Ia-pec            35
TDE                  34
nova                 34
SLSN-II              32
SN Ibn               32
SN Iax               19
SN Ib/c              17
AGN?                 13
SN Ia-CSM            11
SN Ia-SC             10
SN II-pec             7
SN Icn                6
Other                 6
SN Ib-pec             4
LRN                   3
LBV                   3
TDE-He                3
other                 2
ILRT                  2
TDE-H-He              2
Ca-rich               1
SN Ic-pec             1
SN Ca-rich-Ca         1
TDE-featureless       1
Name: count, dtype: int64

39 labels is too many to plot, and most of them are rare. Grouping them by physics, from most detailed to least:

* **`subclass`** (10 groups): SN Ia, SN II, SN Ib/c, SLSN, ...
* **`class3`** (3 groups, used for colouring plots): **SN Ia** / **Other SN** / **Non-SN**
* **`is_ia`** (the ML target): `True` / `False`, or missing if unclassified

A few judgement calls:
* All Ia sub-types (`Ia-91T`, `Ia-91bg`, `Ia-pec`, `Ia-CSM`, `Ia-SC`) count as Ia.
* **`SN Iax` does *not*.** It's a distinct, fainter class that isn't used for cosmology. **Watch out:** `"SN Iax".startswith("SN Ia")` is `True`, so the order of the checks below matters.
* `?` marks a tentative classification (`CV?`, `AGN?`). We keep the class but record the uncertainty.
* `Other` and `other` mean the same thing; real data often has inconsistencies like this.

In [36]:
def to_subclass(label):
    if pd.isna(label):
        return "Unclassified"
    t = label.rstrip("?")
    if t == "SN Iax" or "Ca-rich" in t:     # check BEFORE the generic "SN Ia" test
        return "SN peculiar"
    if t.startswith("SN Ia"):
        return "SN Ia"
    if t.startswith("SN II"):
        return "SN II"
    if t.startswith("SN I"):                 # Ib, Ic, Ic-BL, Ibn, Icn, Ib/c ...
        return "SN Ib/c"
    if t.startswith("SLSN"):
        return "SLSN"
    if t.startswith("TDE"):
        return "TDE"
    if t in ("CV", "nova"):
        return "CV / nova"
    if t == "AGN":
        return "AGN"
    return "Other"                           # LBV, LRN, ILRT, Other, other


df["subclass"] = df["type"].map(to_subclass)
df["type_uncertain"] = df["type"].str.endswith("?").fillna(False)

SN_GROUPS = {"SN Ia": "SN Ia", "SN II": "Other SN", "SN Ib/c": "Other SN",
             "SLSN": "Other SN", "SN peculiar": "Other SN",
             "TDE": "Non-SN", "CV / nova": "Non-SN", "AGN": "Non-SN", "Other": "Non-SN"}
df["class3"] = pd.Categorical(df["subclass"].map(SN_GROUPS),          # Unclassified -> NaN
                              categories=["SN Ia", "Other SN", "Non-SN"], ordered=True)

df["is_ia"] = df["class3"].eq("SN Ia").astype("boolean")
df.loc[df["class3"].isna(), "is_ia"] = pd.NA

df["subclass"].value_counts()

subclass
SN Ia           5309
Unclassified    3370
SN II           1256
CV / nova        514
SN Ib/c          397
AGN              184
SLSN              97
TDE               40
SN peculiar       21
Other             16
Name: count, dtype: int64

**Always check a mapping.** A cross-tab of original label against new class shows that every label went where it should (only the non-zero rows are shown):

In [37]:
check = pd.crosstab(df["type"].fillna("(none)"), df["subclass"])
check.where(check > 0, "")

subclass,AGN,CV / nova,Other,SLSN,SN II,SN Ia,SN Ib/c,SN peculiar,TDE,Unclassified
type,,,,,,,,,,
(none),,,,,,,,,,3370
AGN,171,,,,,,,,,
AGN?,13,,,,,,,,,
CV,,389,,,,,,,,
CV?,,91,,,,,,,,
Ca-rich,,,,,,,,1,,
ILRT,,,2,,,,,,,
LBV,,,3,,,,,,,
LRN,,,3,,,,,,,


### A first look at the classes with `groupby`

`groupby` splits the table into groups, applies a function to each group, and combines the results. `agg` lets us name each output column:

In [38]:
summary = (
    df.groupby("subclass")
      .agg(n=("ZTFID", "size"),
           median_peakmag=("peakmag", "median"),
           median_rise=("rise", "median"),
           median_fade=("fade", "median"),
           median_peakabs=("peakabs", "median"),
           median_redshift=("redshift", "median"),
           median_abs_b=("b", lambda b: b.abs().median()))
      .sort_values("n", ascending=False)
      .round(2)
)
summary

,n,median_peakmag,median_rise,median_fade,median_peakabs,median_redshift,median_abs_b
subclass,,,,,,,
SN Ia,5309,18.26,9.42,14.05,-19.22,0.06,40.25
Unclassified,3370,18.77,8.98,13.94,NaN,NaN,39.44
SN II,1256,18.27,11.89,24.79,-17.73,0.03,40.87
CV / nova,514,17.32,1.00,5.93,-6.98,0.00,15.59
SN Ib/c,397,18.25,10.75,12.13,-17.64,0.03,42.12
AGN,184,18.18,30.18,27.87,-18.84,0.06,45.83
SLSN,97,18.47,28.75,30.48,-21.01,0.15,40.87
TDE,40,18.14,18.28,33.09,-19.59,0.07,39.00
SN peculiar,21,18.29,10.82,9.95,-18.04,0.03,53.95


Some patterns are already visible in this table:

* **SN Ia** peak at an absolute magnitude of about **−19.2**. That's why they're useful "standard candles" for measuring distances.
* Ia are **farther away** (median z ≈ 0.065) than SN II (z ≈ 0.03), because they're brighter and so stay visible to larger distances.
* **SLSNe** (superluminous supernovae) are about 2 magnitudes brighter than Ia and evolve much more slowly.
* **CVs/novae** lie close to the Galactic plane (small |b|) because they're in our own Milky Way. Supernovae are in other galaxies and avoid the dusty plane.

Notebook 2 turns these into plots.

## 10 · Tidy up and save

Putting the columns in a sensible order and saving as **Parquet**. Unlike CSV, Parquet keeps the data types: dates stay dates, booleans stay booleans and categories stay categories. The next notebook can load it and start straight away.

In [39]:
df["peakfilt"] = df["peakfilt"].astype("category")

column_order = [
    "ZTFID", "IAUID", "discID",
    "type", "subclass", "class3", "is_ia", "type_uncertain",
    "discdate", "peakdate", "peakt", "disc_to_peak_days",
    "RA", "Dec", "ra_deg", "dec_deg", "b", "A_V",
    "peakfilt", "peakmag", "peakabs",
    "rise", "rise_is_limit", "fade", "fade_is_limit", "duration", "duration_is_limit",
    "redshift", "hostabs", "hostcol",
]
clean = df[column_order].reset_index(drop=True)

from pathlib import Path
Path("data").mkdir(exist_ok=True)
clean.to_parquet("data/bts_clean.parquet", index=False)
clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 11204 entries, 0 to 11203
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ZTFID              11204 non-null  str           
 1   IAUID              11097 non-null  str           
 2   discID             8317 non-null   str           
 3   type               7834 non-null   str           
 4   subclass           11204 non-null  str           
 5   class3             7834 non-null   category      
 6   is_ia              7834 non-null   boolean       
 7   type_uncertain     11204 non-null  bool          
 8   discdate           11204 non-null  datetime64[us]
 9   peakdate           11204 non-null  datetime64[ns]
 10  peakt              11204 non-null  float64       
 11  disc_to_peak_days  11204 non-null  float64       
 12  RA                 11204 non-null  str           
 13  Dec                11204 non-null  str           
 14  ra_deg           

## Summary: what I learned

**About pandas**
* `read_csv(na_values=...)` handles non-standard missing-value markers. Check `dtypes` immediately: text where you expected numbers means something odd is in the column.
* The `.str` and `.dt` accessors apply string and date operations to a whole column at once, with no loops.
* Boolean masks combined with `&`, `|`, `~` (each condition in parentheses) are the main way to filter rows.
* `groupby` + `agg` and `crosstab` turn a table into summaries that answer questions.
* Parquet keeps data types; CSV doesn't.

**About this dataset**
* 11,217 rows → 11,204 unique objects after removing 13 duplicates.
* ~30% are unclassified. Of the classified objects, ~68% are SN Ia.
* Rise/fade/duration are often **lower limits** (censored data). We kept them, with a flag.
* **Data error found:** 33 rows had wrong `b` and `A_V` values (23 of them all identical). Recomputing `b` from RA/Dec exposed it.
* **Redshift exists only for classified objects**, a warning sign for later ML work.
* **2026 rows have no types in this export** even though most are already named `SN...`. The IAU prefix leaks the label.
* The catalogue stops at magnitude 19. It's a *magnitude-limited* sample.

## Exercises
1. How many SN Ia were discovered in each year? (Hint: `df[df.is_ia == True].groupby(df.discdate.dt.year).size()`)
2. Which object has the longest `duration` that is **not** a lower limit? What type is it?
3. Find the CV classified with the largest |b|. Is it really in the Milky Way's plane?
4. Using `pd.cut`, bin `redshift` into `[0, 0.02, 0.05, 0.1, 0.2, 0.5]` and count the subclasses in each bin with `pd.crosstab`. What fraction is Ia in each bin?

In [40]:
df[df.is_ia == True].groupby(df.discdate.dt.year).size()    

discdate
2017      3
2018    240
2019    779
2020    869
2021    857
2022    680
2023    566
2024    698
2025    617
dtype: int64